In [10]:
!pip install fastmcp


In [16]:
import json
from datetime import date, datetime
from pathlib import Path
import nest_asyncio
import asyncio

from fastmcp import FastMCP

nest_asyncio.apply()

mcp = FastMCP("Tarefas da Faculdade")

ARQUIVO = Path("tarefas.json")
PRIORIDADES = ["baixa", "media", "alta"]


def carregar_tarefas():
    if not ARQUIVO.exists():
        return []
    try:
        return json.loads(ARQUIVO.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return []


def salvar_tarefas(tarefas):
    ARQUIVO.write_text(json.dumps(tarefas, ensure_ascii=False, indent=2), encoding="utf-8")


def buscar_tarefa(tarefas, id_tarefa):
    for t in tarefas:
        if t["id"] == id_tarefa:
            return t
    raise ValueError(f"Nao achei nenhuma tarefa com id {id_tarefa}")


def dias_restantes(prazo):
    return (datetime.strptime(prazo, "%Y-%m-%d").date() - date.today()).days


@mcp.tool
def adicionar_tarefa(titulo, materia, prazo, prioridade="media"):
    titulo = titulo.strip()
    if not titulo:
        raise ValueError("O titulo nao pode ficar vazio")

    prioridade = prioridade.lower()
    if prioridade not in PRIORIDADES:
        raise ValueError(f"Prioridade invalida. Use: {', '.join(PRIORIDADES)}")

    try:
        datetime.strptime(prazo, "%Y-%m-%d")
    except ValueError:
        raise ValueError("Prazo tem que estar no formato AAAA-MM-DD, tipo 2026-11-20")

    tarefas = carregar_tarefas()
    novo_id = max((t["id"] for t in tarefas), default=0) + 1

    tarefa = {
        "id": novo_id,
        "titulo": titulo,
        "materia": materia.strip(),
        "prazo": prazo,
        "prioridade": prioridade,
        "concluida": False,
    }
    tarefas.append(tarefa)
    salvar_tarefas(tarefas)
    return tarefa


@mcp.tool
def listar_tarefas(materia=None, mostrar_concluidas=False):
    tarefas = carregar_tarefas()

    if not mostrar_concluidas:
        tarefas = [t for t in tarefas if not t["concluida"]]
    if materia:
        tarefas = [t for t in tarefas if t["materia"].lower() == materia.lower()]

    for t in tarefas:
        t["dias_restantes"] = dias_restantes(t["prazo"])

    return sorted(tarefas, key=lambda t: t["prazo"])


@mcp.tool
def concluir_tarefa(id_tarefa):
    tarefas = carregar_tarefas()
    tarefa = buscar_tarefa(tarefas, id_tarefa)

    if tarefa["concluida"]:
        return f"A tarefa '{tarefa['titulo']}' ja estava concluida"

    tarefa["concluida"] = True
    salvar_tarefas(tarefas)
    return f"Boa! '{tarefa['titulo']}' marcada como concluida"


@mcp.tool
def remover_tarefa(id_tarefa):
    tarefas = carregar_tarefas()
    tarefa = buscar_tarefa(tarefas, id_tarefa)
    tarefas.remove(tarefa)
    salvar_tarefas(tarefas)
    return f"Tarefa '{tarefa['titulo']}' removida"


@mcp.tool
def tarefas_atrasadas():
    return [t for t in listar_tarefas() if t["dias_restantes"] < 0]


@mcp.resource("tarefas://resumo")
def resumo():
    tarefas = carregar_tarefas()
    pendentes = [t for t in tarefas if not t["concluida"]]
    return {
        "total": len(tarefas),
        "pendentes": len(pendentes),
        "concluidas": len(tarefas) - len(pendentes),
        "atrasadas": len([t for t in pendentes if dias_restantes(t["prazo"]) < 0]),
    }


@mcp.resource("tarefas://{id_tarefa}")
def detalhe_tarefa(id_tarefa):
    return buscar_tarefa(carregar_tarefas(), int(id_tarefa))


@mcp.prompt
def planejar_semana(horas_por_dia=2):
    pendentes = listar_tarefas()
    if not pendentes:
        return "Nao tenho tarefas pendentes. Me sugere o que revisar essa semana?"

    lista = "\n".join(
        f"- {t['titulo']} ({t['materia']}) | prazo {t['prazo']} | prioridade {t['prioridade']}"
        for t in pendentes
    )
    return (
        f"Tenho {horas_por_dia} horas por dia pra estudar. Essas sao minhas tarefas pendentes:\n"
        f"{lista}\n\n"
        "Monta um plano de segunda a domingo, priorizando o que vence antes e o que e prioridade alta."
    )


if __name__ == "__main__":
    loop = asyncio.get_event_loop()
    loop.create_task(mcp.run_async())

In [14]:
import asyncio
await asyncio.sleep(1)

print("--- Testando o Servidor MCP ---")

try:
    print("\nAdicionando tarefa...")
    nova = adicionar_tarefa(
        titulo="Estudar para a prova de Cálculo",
        materia="Cálculo I",
        prazo="2026-11-20",
        prioridade="alta"
    )
    print("Tarefa criada:", nova)

    print("\nListando tarefas pendentes:")
    tarefas = listar_tarefas()
    for t in tarefas:
        print(f"- [{t['prioridade'].upper()}] {t['titulo']} ({t['materia']}) - Vence em: {t['prazo']} (Faltam {t['dias_restantes']} dias)")

    print("\nConsultando o recurso de resumo:")
    res = resumo()
    print("Resumo Geral:", res)

except Exception as e:
    print(f"Ocorreu um erro durante os testes: {e}")

--- Testando o Servidor MCP ---

Adicionando tarefa...
Tarefa criada: {'id': 2, 'titulo': 'Estudar para a prova de Cálculo', 'materia': 'Cálculo I', 'prazo': '2026-11-20', 'prioridade': 'alta', 'concluida': False}

Listando tarefas pendentes:
- [ALTA] Estudar para a prova de Cálculo (Cálculo I) - Vence em: 2026-11-20 (Faltam 18 dias)
- [ALTA] Estudar para a prova de Cálculo (Cálculo I) - Vence em: 2026-11-20 (Faltam 18 dias)

Consultando o recurso de resumo:
Resumo Geral: {'total': 2, 'pendentes': 2, 'concluidas': 0, 'atrasadas': 0}
